## a) Import Needed Libraries

In [ ]:
# Basics
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

# Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample

# Metrics
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score

from imblearn.over_sampling import SMOTE

## b) Read the Data and Explore

In [ ]:
# Read data
df = pd.read_csv('iris-data-new2.csv')

# Check data
print(df.head())

In [ ]:
print(df.describe())

In [ ]:
print(df.info())

In [ ]:
print(df['price'].value_counts())

In [ ]:
print(df.isnull().sum())

## c) Decision Tree Model (only 4 numeric features)

In [ ]:
# Define features and target
features = ['sepal_length_cm', 'sepal_width_cm', 'petal_length_cm', 'petal_width_cm']
X = df[features]
y = df['price']

# Split data (75% training, 25% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

## d) Analyze metrics for Decision Tree

In [ ]:
# Metrics
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Precision:", precision_score(y_test, y_pred_dt, average='weighted'))
print("Recall:", recall_score(y_test, y_pred_dt, average='weighted'))
print("F1-Score:", f1_score(y_test, y_pred_dt, average='weighted'))

## e) Confusion Matrix for Decision Tree

In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred_dt)
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Decision Tree')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

f) Nearest Neighbor Model

In [ ]:
# KNN
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

g) Analyze metrics for KNN

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_knn))
print("Precision:", precision_score(y_test, y_pred_knn, average='weighted'))
print("Recall:", recall_score(y_test, y_pred_knn, average='weighted'))
print("F1-Score:", f1_score(y_test, y_pred_knn, average='weighted'))

# Confusion Matrix
conf_matrix_knn = confusion_matrix(y_test, y_pred_knn)
sns.heatmap(conf_matrix_knn, annot=True, fmt='d', cmap='Greens')
plt.title('Confusion Matrix - KNN')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## h) Balance Dataset

### Method 1: Oversampling (Simple Resampling)

In [ ]:
# Separate majority and minority classes
low = df[df['price'] == 'Low']
medium = df[df['price'] == 'Medium']
high = df[df['price'] == 'High']

# Find the maximum size
max_size = max(len(low), len(medium), len(high))

# Upsample
low_upsampled = resample(low, replace=True, n_samples=max_size, random_state=42)
medium_upsampled = resample(medium, replace=True, n_samples=max_size, random_state=42)
high_upsampled = resample(high, replace=True, n_samples=max_size, random_state=42)

# Combine
df_balanced = pd.concat([low_upsampled, medium_upsampled, high_upsampled])

# Shuffle
df_balanced = df_balanced.sample(frac=1, random_state=42)

### Method 2: SMOTE (Synthetic Minority Oversampling Technique)

In [ ]:
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X, y)

## i) Fit Nearest Neighbor, Decision Trees, SVM, Neural Networks (Balanced Dataset)

In [ ]:
# Encode 'iris_type' and 'package'
le_iris = LabelEncoder()
le_package = LabelEncoder()

df_balanced['iris_type_encoded'] = le_iris.fit_transform(df_balanced['iris_type'])
df_balanced['package_encoded'] = le_package.fit_transform(df_balanced['package'])

# Define features and target
X_full = df_balanced[['sepal_length_cm', 'sepal_width_cm', 'petal_length_cm', 'petal_width_cm', 'iris_type_encoded', 'package_encoded']]
y_full = df_balanced['price']

# Train-test split
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(X_full, y_full, test_size=0.25, random_state=42)

# Nearest Neighbor
knn_full = KNeighborsClassifier()
knn_full.fit(X_train_full, y_train_full)

# Decision Tree
dt_full = DecisionTreeClassifier()
dt_full.fit(X_train_full, y_train_full)

# SVM
svm_full = SVC()
svm_full.fit(X_train_full, y_train_full)

# Neural Network
mlp_full = MLPClassifier(max_iter=10000, random_state=42)
mlp_full.fit(X_train_full, y_train_full)

## j) Analyze Metrics + Confusion Matrices for Each Model

In [ ]:
# Predict
y_pred_knn_full = knn_full.predict(X_test_full)

# Metrics
print("Nearest Neighbor")
print("Accuracy:", accuracy_score(y_test_full, y_pred_knn_full))
print("Precision:", precision_score(y_test_full, y_pred_knn_full, average='weighted'))
print("Recall:", recall_score(y_test_full, y_pred_knn_full, average='weighted'))
print("F1-Score:", f1_score(y_test_full, y_pred_knn_full, average='weighted'))

# Confusion Matrix
sns.heatmap(confusion_matrix(y_test_full, y_pred_knn_full), annot=True, fmt='d', cmap='Greens')
plt.title('Confusion Matrix - Nearest Neighbor')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# Predict
y_pred_dt_full = dt_full.predict(X_test_full)

# Metrics
print("Decision Tree")
print("Accuracy:", accuracy_score(y_test_full, y_pred_dt_full))
print("Precision:", precision_score(y_test_full, y_pred_dt_full, average='weighted'))
print("Recall:", recall_score(y_test_full, y_pred_dt_full, average='weighted'))
print("F1-Score:", f1_score(y_test_full, y_pred_dt_full, average='weighted'))

# Confusion Matrix
sns.heatmap(confusion_matrix(y_test_full, y_pred_dt_full), annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Decision Tree')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# Predict
y_pred_svm_full = svm_full.predict(X_test_full)

# Metrics
print("SVM")
print("Accuracy:", accuracy_score(y_test_full, y_pred_svm_full))
print("Precision:", precision_score(y_test_full, y_pred_svm_full, average='weighted'))
print("Recall:", recall_score(y_test_full, y_pred_svm_full, average='weighted'))
print("F1-Score:", f1_score(y_test_full, y_pred_svm_full, average='weighted'))

# Confusion Matrix
sns.heatmap(confusion_matrix(y_test_full, y_pred_svm_full), annot=True, fmt='d', cmap='Purples')
plt.title('Confusion Matrix - SVM')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# Predict
y_pred_mlp_full = mlp_full.predict(X_test_full)

# Metrics
print("Neural Network")
print("Accuracy:", accuracy_score(y_test_full, y_pred_mlp_full))
print("Precision:", precision_score(y_test_full, y_pred_mlp_full, average='weighted'))
print("Recall:", recall_score(y_test_full, y_pred_mlp_full, average='weighted'))
print("F1-Score:", f1_score(y_test_full, y_pred_mlp_full, average='weighted'))

# Confusion Matrix
sns.heatmap(confusion_matrix(y_test_full, y_pred_mlp_full), annot=True, fmt='d', cmap='Oranges')
plt.title('Confusion Matrix - Neural Network')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## k) 5-Fold Cross-Validation for All Models

In [ ]:
# Nearest Neighbor
scores_knn = cross_val_score(knn_full, X_full, y_full, cv=5, scoring='accuracy')
print('KNN 5-Fold CV Accuracy:', np.mean(scores_knn))

# Decision Tree
scores_dt = cross_val_score(dt_full, X_full, y_full, cv=5, scoring='accuracy')
print('Decision Tree 5-Fold CV Accuracy:', np.mean(scores_dt))

# SVM
scores_svm = cross_val_score(svm_full, X_full, y_full, cv=5, scoring='accuracy')
print('SVM 5-Fold CV Accuracy:', np.mean(scores_svm))

# Neural Network
scores_mlp = cross_val_score(mlp_full, X_full, y_full, cv=5, scoring='accuracy')
print('Neural Network 5-Fold CV Accuracy:', np.mean(scores_mlp))

## l) Grid Search for Best Parameters (Top 2 models)

In [ ]:
param_grid_dt = {
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

grid_dt = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid_dt, cv=5, scoring='accuracy')
grid_dt.fit(X_full, y_full)

print("Best parameters for Decision Tree:", grid_dt.best_params_)

In [ ]:
param_grid_mlp = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'sgd'],
    'alpha': [0.0001, 0.001],
    'learning_rate': ['constant', 'adaptive']
}

grid_mlp = GridSearchCV(mlp_full, param_grid_mlp, cv=5, scoring='accuracy')
grid_mlp.fit(X_full, y_full)

print("Best parameters for Neural Network:", grid_mlp.best_params_)

## m) Final Training and Evaluation (With your best parameters)

In [ ]:
# Best Decision Tree with found parameters
best_dt = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=7,
    min_samples_split=2,
    random_state=42
)

# Best Neural Network with found parameters
best_mlp = MLPClassifier(
    activation='tanh',
    alpha=0.0001,
    hidden_layer_sizes=(100, 50),
    learning_rate='constant',
    solver='adam',
    max_iter=1000,
    random_state=42
)

# New Train-Test Split
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(X_full, y_full, test_size=0.25, random_state=42)

# Train both models
best_dt.fit(X_train_final, y_train_final)
best_mlp.fit(X_train_final, y_train_final)

# Predictions
y_pred_dt_final = best_dt.predict(X_test_final)
y_pred_mlp_final = best_mlp.predict(X_test_final)

In [ ]:
print("Final Model: Decision Tree")
print("Accuracy:", accuracy_score(y_test_final, y_pred_dt_final))
print("Precision:", precision_score(y_test_final, y_pred_dt_final, average='weighted'))
print("Recall:", recall_score(y_test_final, y_pred_dt_final, average='weighted'))
print("F1-Score:", f1_score(y_test_final, y_pred_dt_final, average='weighted'))

# Confusion Matrix
sns.heatmap(confusion_matrix(y_test_final, y_pred_dt_final), annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Best Decision Tree')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
print("Final Model: Neural Network")
print("Accuracy:", accuracy_score(y_test_final, y_pred_mlp_final))
print("Precision:", precision_score(y_test_final, y_pred_mlp_final, average='weighted'))
print("Recall:", recall_score(y_test_final, y_pred_mlp_final, average='weighted'))
print("F1-Score:", f1_score(y_test_final, y_pred_mlp_final, average='weighted'))

# Confusion Matrix
sns.heatmap(confusion_matrix(y_test_final, y_pred_mlp_final), annot=True, fmt='d', cmap='Oranges')
plt.title('Confusion Matrix - Best Neural Network')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# Create a comparison table
results = {
    'Model': ['Decision Tree', 'Neural Network'],
    'Accuracy': [
        accuracy_score(y_test_final, y_pred_dt_final),
        accuracy_score(y_test_final, y_pred_mlp_final)
    ],
    'Precision': [
        precision_score(y_test_final, y_pred_dt_final, average='weighted'),
        precision_score(y_test_final, y_pred_mlp_final, average='weighted')
    ],
    'Recall': [
        recall_score(y_test_final, y_pred_dt_final, average='weighted'),
        recall_score(y_test_final, y_pred_mlp_final, average='weighted')
    ],
    'F1-Score': [
        f1_score(y_test_final, y_pred_dt_final, average='weighted'),
        f1_score(y_test_final, y_pred_mlp_final, average='weighted')
    ]
}

df_results = pd.DataFrame(results)
print(df_results)